# Practice 41 — pandas + NumPy

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: `.str.contains()` and `.str.extract()`

pandas `.str` methods let you work with string columns without loops or `.apply()`.

**`.str.contains(pattern)`** — returns a boolean Series: True where the string matches the pattern.
```python
df['name'].str.contains('ford')          # exact substring match
df['name'].str.contains('ford', case=False)   # case-insensitive
df['name'].str.contains(r'^toyota')      # regex: starts with 'toyota'
```

**`.str.extract(pattern)`** — pulls out the part of the string that matches a regex group `()`.
```python
# Extract digits from strings like 'Model-42', 'Item-7'
df['col'].str.extract(r'(\d+)')          # returns a DataFrame with one column
df['col'].str.extract(r'(\d+)')[0]       # get the Series

# Extract the word before a dash
df['col'].str.extract(r'^(\w+)-')        # captures word at the start before '-'
```

Both accept regular expressions. Key regex basics:
- `\d` — any digit, `\w` — any word character (letter/digit/underscore)
- `+` — one or more, `*` — zero or more
- `^` — start of string, `$` — end of string
- `( )` — capture group (required for `.str.extract()`)

---
### Task: mpg name column

Load `sns.load_dataset('mpg')`. The `name` column has values like `'chevrolet chevelle malibu'`, `'ford torino'`, `'toyota corolla'`.

1. Use `.str.contains()` to find all cars made by `'ford'` (case-insensitive). How many are there?
2. Use `.str.extract()` to pull out the **first word** of each car name as a new column `make`. (Hint: `r'^(\w+)'`)
3. Which `make` appears most often? Use `.value_counts()` — no groupby needed.
4. Use `.str.contains()` to flag cars whose name contains `'pickup'` or `'truck'` — one expression using `|` inside the regex pattern. How many?
5. Convert `make` to `category`. How much memory does it save vs `object` dtype?

In [ ]:
# Your code here

---
## Level 2 — titanic: string + vectorized

Load `sns.load_dataset('titanic')`. No `.apply()`.

The `deck` column contains letter codes like `'A'`, `'B'`, `'C'` etc (with many nulls).

1. Use `.str.contains()` to flag passengers whose `deck` is in the upper decks (`'A'`, `'B'`, or `'C'`). Handle NaN with `na=False`.
2. The `embarked` column has codes `'S'`, `'C'`, `'Q'`. Use `.str.extract()` or `np.select()` to add a `port` column with full names: `'Southampton'`, `'Cherbourg'`, `'Queenstown'`.
3. Add `title` by extracting the title from the `name` column using `.str.extract()`. Titles look like `'Mr.'`, `'Mrs.'`, `'Miss.'`, `'Master.'` — pattern: `r'([A-Z][a-z]+\.)` 
4. Use `.str.contains()` to flag passengers with a rare title (not Mr, Mrs, Miss, or Master). What fraction of them survived?
5. Build a pivot table: mean `survived` by `title` × `pclass`, `observed=True`. Stack it and find the top combination.

In [ ]:
# Your code here

---
## Level 3 — pipeline with string ops

You have a product dataset below with messy string columns. Write a `.pipe()` chain — no `.apply()`.

- **`clean(df)`** — use `.str.extract()` to pull the numeric part from `sku` into a new column `sku_num` (as integer). Use `.str.contains()` to add `is_sale`: True where `tags` contains `'sale'` (case-insensitive). Convert `category` to `category` dtype.
- **`enrich(df)`** — add `final_price`: if `is_sale`, apply a 20% discount (price × 0.8), otherwise keep original price — vectorized using `np.where`. Add `margin_class` with `np.select()`: `'high'` (margin > 0.4), `'mid'` (margin > 0.2), `'low'` otherwise.

After the chain:
- What is the mean `final_price` per `category`? One groupby.
- Use a list comprehension with `zip` to collect product names where `is_sale` is True.
- Use `.str.contains()` to find products whose `name` contains a digit — any product names that look like model numbers?

In [ ]:
products = pd.DataFrame({
    'name':     ['Widget Pro', 'Gadget X200', 'Doohickey', 'Thingamajig 3', 'Gizmo',
                 'Widget Lite', 'Gadget X100', 'Contraption', 'Widget Max', 'Doohickey Plus'],
    'sku':      ['WG-001', 'GD-002', 'DH-003', 'TM-004', 'GZ-005',
                 'WG-006', 'GD-007', 'CT-008', 'WG-009', 'DH-010'],
    'category': ['Widgets','Gadgets','Doohickeys','Thingamajigs','Gizmos',
                 'Widgets','Gadgets','Contraptions','Widgets','Doohickeys'],
    'price':    [29.99, 149.99, 9.99, 49.99, 19.99, 14.99, 99.99, 79.99, 59.99, 24.99],
    'margin':   [0.45, 0.30, 0.15, 0.38, 0.22, 0.50, 0.28, 0.18, 0.42, 0.20],
    'tags':     ['bestseller', 'sale,new', 'sale', 'new', 'bestseller',
                 'sale,bestseller', 'new', 'sale', 'bestseller', 'new']
})

# Your code here